In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv("C:\\Users\\delta\\Predictive-maintenence-iot\\dataset\\ai4i2020.csv")
print("Dataset Loaded Successfully")
print(df.shape)

In [ ]:
target = "Machine failure"
X = df.drop(columns=["UDI","Product ID",target])
y = df[target]
X = pd.get_dummies(X,columns=["Type"],drop_first=True)
print("Feature Shape :", X.shape)
print("Target Shape :", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print("Training Samples :", len(X_train))
print("Testing Samples :", len(X_test))

In [ ]:
pipeline = Pipeline([("smote",SMOTE(random_state=42)),("model",LogisticRegression(max_iter=1000,random_state=42))])
pipeline.fit(X_train,y_train)
print("Pipeline Trained Successfully")

In [ ]:
predictions = pipeline.predict(X_test)
print("Predictions Generated")
print(predictions[:20])

In [ ]:
prediction_probabilities = pipeline.predict_proba(X_test)
print("Prediction Probabilities")
prediction_probabilities[:10]

In [ ]:
failure_probability = prediction_probabilities[:,1]
print(failure_probability[:20])

In [ ]:
results = pd.DataFrame({
    "Actual":y_test.values,
    "Predicted":predictions,
    "Failure Probability":failure_probability
})
results.head(20)

In [ ]:
low_risk = results.sort_values(by="Failure Probability")
low_risk.head(15)

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(failure_probability,bins=25,edgecolor="black")
plt.title("Distribution of Failure Probability")
plt.xlabel("Failure Probability")
plt.ylabel("Number of Machines")
plt.savefig("failure_probability_distribution.png")
plt.show()

In [ ]:
results["Risk Level"] = pd.cut(
    results["Failure Probability"],
    bins=[0,0.25,0.50,0.75,1],
    labels=["Low","Medium","High","Very High"])
results.head()

In [ ]:
risk_summary = results["Risk Level"].value_counts()
print(risk_summary)

In [ ]:
plt.figure(figsize=(7,5))
risk_summary.plot(kind="bar")
plt.title("Predicted Risk Levels")
plt.ylabel("Number of Machines")
plt.savefig("predicted_risk_levels.png")
plt.show()

In [ ]:
accuracy = accuracy_score(y_test,predictions)
precision = precision_score(y_test,predictions)
recall = recall_score(y_test,predictions)
f1 = f1_score(y_test,predictions)
metrics = pd.DataFrame({
    "Metric":["Accuracy","Precision","Recall","F1 Score"],
    "Score":[accuracy,precision,recall,f1]})
metrics

In [ ]:
print("Probability Statistics")
print("="*50)
print(results["Failure Probability"].describe())

In [ ]:
confident_failures = results[results["Predicted"]==1]
confident_failures.sort_values(by="Failure Probability",ascending=False).head(10)

In [ ]:
report = f"""
======================================
Week 4 Day 1 Report
======================================
Task Completed:
Extracted prediction probabilities

Generated prediction table

Classified machine risk levels

Reviewed model confidence

Prepared dataset for Precision-Recall Curve

Average Failure Probability:
{results["Failure Probability"].mean():.3f}

Highest Probability:
{results["Failure Probability"].max():.3f}

Lowest Probability:
{results["Failure Probability"].min():.3f}

The extracted probability scores will be used in the next phase to generate the Precision-Recall Curve and calculate Average Precision Score.
======================================
"""
print(report)